# Eksplorasi PySpark — NovelLens

Notebook untuk memeriksa data lake secara interaktif. Berjalan di container `notebook`,
yang menjadi **driver Spark tersendiri** — terpisah dari container `pipeline` yang dipakai
job batch, sehingga eksperimen di sini tidak mengganggu jalannya pipeline.

Buka lewat <http://localhost:8888>.

| Zona | Isi |
|---|---|
| `s3a://raw-data/` | CSV mentah apa adanya (Fase 1) |
| `s3a://clean-data/novels_clean` | Parquet bersih siap analisis (Fase 2) |
| `s3a://curated-data/analytics/` | Hasil kueri Spark SQL (Fase 3) |
| `s3a://curated-data/models/` | Cadangan artefak model (Fase 4) |

## 1. Menyalakan SparkSession

Selalu lewat `build_spark_session()`, jangan `SparkSession.builder` langsung — factory itu
yang memasang konfigurasi konektor S3A ke MinIO dan `spark.driver.host` yang benar.
Tanpa keduanya, job menggantung di stage pertama atau gagal dengan `403 Forbidden`.

In [1]:
from src.common.config import get_settings
from src.common.spark_session import build_spark_session

settings = get_settings()
spark = build_spark_session("notebook")

print(f"Spark   : {spark.version}")
print(f"Master  : {settings.spark.master_url}")
print(f"MinIO   : {settings.minio.endpoint}")
print("Spark UI: http://localhost:8080")

Picked up JAVA_TOOL_OPTIONS: -Djavax.net.ssl.trustStore=/certs/truststore.p12 -Djavax.net.ssl.trustStorePassword=changeit -Djavax.net.ssl.trustStoreType=PKCS12


Picked up JAVA_TOOL_OPTIONS: -Djavax.net.ssl.trustStore=/certs/truststore.p12 -Djavax.net.ssl.trustStorePassword=changeit -Djavax.net.ssl.trustStoreType=PKCS12


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/15 08:42:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


2026-10-15 08:42:29 | INFO     | src.common.spark_session | SparkSession aktif | app=notebook | master=spark://spark-master:7077 | versi=3.5.3


Spark   : 3.5.3
Master  : spark://spark-master:7077
MinIO   : https://minio:9000
Spark UI: http://localhost:8080


## 2. Isi data lake

In [2]:
from src.common.storage import list_objects

for bucket in settings.minio.buckets:
    objects = list_objects(bucket)
    total_mb = sum(o["Size"] for o in objects) / 1_048_576
    print(f"{bucket:<14} {len(objects):>4} objek  {total_mb:>8.2f} MB")

raw-data          4 objek     58.83 MB
clean-data        3 objek    110.96 MB
curated-data    116 objek    311.19 MB


## 3. Membaca data bersih

`novels_clean` adalah keluaran Fase 2: sentinel sudah dinormalkan jadi NULL, duplikat dibuang
berdasarkan `series_id`, sinopsis sudah tergabung, dan korpus teks sudah ditokenisasi.

In [3]:
clean_uri = settings.minio.s3a_uri(settings.minio.bucket_clean, "novels_clean")
novels = spark.read.parquet(clean_uri).cache()
novels.createOrReplaceTempView("novels")

print(f"{novels.count():,} baris x {len(novels.columns)} kolom")
novels.select(
    "title", "primary_genre", "chapter_count", "rating_score",
    "weighted_rating", "synopsis_word_count", "token_count",
).show(8, truncate=32)

26/10/15 08:42:30 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


26/10/15 08:42:34 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


39,668 baris x 50 kolom


+--------------------------+-------------+-------------+------------+---------------+-------------------+-----------+
|                     title|primary_genre|chapter_count|rating_score|weighted_rating|synopsis_word_count|token_count|
+--------------------------+-------------+-------------+------------+---------------+-------------------+-----------+
|          Against the Gods|       Action|         2200|         3.9|         3.8993|                131|        180|
|    Stellar Transformation|       Action|          680|         4.2|         4.1957|                117|        132|
|      The Bathroom Goddess|    Adventure|           20|         4.7|         4.6791|                106|         69|
|  The Legend of Sun Knight|       Action|          104|         4.7|         4.6844|                193|        166|
|Young God Divine Armaments|       Action|          215|         3.5|         3.5075|                169|        142|
|                       Ark|       Action|          600|

In [4]:
novels.printSchema()

root
 |-- series_id: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- url: string (nullable = true)
 |-- type: string (nullable = true)
 |-- type_origin: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- language: string (nullable = true)
 |-- authors: string (nullable = true)
 |-- original_publisher: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- status_coo: string (nullable = true)
 |-- chapter_count: integer (nullable = true)
 |-- is_completed: integer (nullable = true)
 |-- licensed: integer (nullable = true)
 |-- completely_translated: integer (nullable = true)
 |-- release_frequency: string (nullable = true)
 |-- rating_score: double (nullable = true)
 |-- rating_votes: integer (nullable = true)
 |-- activity_week_rank: integer (nullable = true)
 |-- activity_month_rank: integer (nullable = true)
 |-- activity_all_rank: integer (nullable = true)
 |-- reading_list_count: integer (nullable 

## 4. Kueri Spark SQL bebas

Ubah kueri di bawah untuk eksplorasi sendiri. Ingat dua aturan analitik dari
`Doc/data_contract.md`:

1. Pakai `weighted_rating`, bukan `rating_score` mentah, untuk peringkat apa pun.
2. Batasi populasi ke `is_rated_reliable` (>= 10 suara) saat menghitung rata-rata rating.

In [5]:
spark.sql("""
    SELECT genre,
           COUNT(*)                        AS n_novels,
           ROUND(AVG(weighted_rating), 4)  AS avg_rating,
           ROUND(AVG(chapter_count), 1)    AS avg_chapters
    FROM (SELECT explode(genre_list) AS genre, weighted_rating, chapter_count
          FROM novels WHERE is_rated_reliable)
    GROUP BY genre
    HAVING COUNT(*) >= 25
    ORDER BY avg_rating DESC
    LIMIT 15
""").show(truncate=False)

+-------------+--------+----------+------------+
|genre        |n_novels|avg_rating|avg_chapters|
+-------------+--------+----------+------------+
|Shounen Ai   |1333    |3.9035    |115.3       |
|Yaoi         |3875    |3.8789    |127.0       |
|Historical   |1503    |3.8147    |270.3       |
|Mystery      |2220    |3.8081    |396.6       |
|Horror       |1257    |3.7835    |320.5       |
|Psychological|2375    |3.7769    |267.6       |
|Sports       |293     |3.7612    |235.1       |
|Sci-fi       |1675    |3.7589    |405.8       |
|Slice of Life|5288    |3.7571    |249.7       |
|Supernatural |3929    |3.7545    |360.9       |
|Wuxia        |438     |3.7444    |359.0       |
|Comedy       |8520    |3.735     |306.4       |
|Tragedy      |1864    |3.729     |304.9       |
|Mecha        |444     |3.7147    |308.9       |
|Drama        |8755    |3.7129    |255.1       |
+-------------+--------+----------+------------+



In [6]:
spark.sql(
    """
    select synopsis from novels
    LIMIT 10
    """
).show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## 5. Memeriksa rencana eksekusi

Berguna untuk memastikan Spark benar-benar mem-*pushdown* filter ke Parquet
dan tidak melakukan shuffle yang tidak perlu.

In [7]:
(
    novels.filter("is_rated_reliable AND chapter_count > 500")
    .groupBy("type_origin")
    .avg("weighted_rating")
    .explain("formatted")
)

== Physical Plan ==
AdaptiveSparkPlan (10)
+- HashAggregate (9)
   +- Exchange (8)
      +- HashAggregate (7)
         +- Project (6)
            +- Filter (5)
               +- InMemoryTableScan (1)
                     +- InMemoryRelation (2)
                           +- * ColumnarToRow (4)
                              +- Scan parquet  (3)


(1) InMemoryTableScan
Output [4]: [chapter_count#12, is_rated_reliable#42, type_origin#4, weighted_rating#41]
Arguments: [chapter_count#12, is_rated_reliable#42, type_origin#4, weighted_rating#41], [isnotnull(is_rated_reliable#42), isnotnull(chapter_count#12), is_rated_reliable#42, (chapter_count#12 > 500)]

(2) InMemoryRelation
Arguments: [series_id#0, title#1, url#2, type#3, type_origin#4, genres#5, tags#6, language#7, authors#8, original_publisher#9, year#10, status_coo#11, chapter_count#12, is_completed#13, licensed#14, completely_translated#15, release_frequency#16, rating_score#17, rating_votes#18, activity_week_rank#19, activity_month_ra

## 6. Membaca hasil analitik Fase 3

Tabel-tabel ini yang dibaca dashboard. Daftar lengkapnya diambil dari isi bucket.

In [8]:
prefixes = sorted({
    o["Key"].split("/")[1]
    for o in list_objects(settings.minio.bucket_curated, "analytics/")
})
print(f"{len(prefixes)} tabel analitik:")
print("  " + "\n  ".join(prefixes))

31 tabel analitik:
  best_rated_tags
  correlation_metrics
  decade_trend
  genre_combinations
  genre_frequency
  genre_origin_matrix
  genre_pairs
  genre_ranking
  graph_author_pagerank
  graph_genre_edges
  graph_genre_pagerank
  graph_similarity_communities
  graph_similarity_community_edges
  graph_similarity_edges
  graph_similarity_homophily
  graph_similarity_nodes
  graph_similarity_summary
  graph_summary
  length_vs_rating
  origin_type_summary
  quality_report
  release_pace_vs_rating
  stat_descriptive
  stat_outlier_novels
  stat_outlier_summary
  synopsis_discriminative_tokens
  text_correlation
  text_richness_vs_rating
  top_tags
  top_tokens
  viz_novels


In [9]:
table = "genre_ranking"  # ganti dengan nama tabel mana pun di atas
uri = settings.minio.s3a_uri(settings.minio.bucket_curated, f"analytics/{table}")
spark.read.parquet(uri).show(12, truncate=False)

+-------------+--------+-------------------+------------+---------------+--------------+--------------+
|genre        |n_novels|avg_weighted_rating|avg_chapters|median_chapters|rank_by_rating|rank_by_length|
+-------------+--------+-------------------+------------+---------------+--------------+--------------+
|Shounen Ai   |1333    |3.9035             |115.3       |90             |1             |35            |
|Yaoi         |3875    |3.8789             |127.0       |102            |2             |33            |
|Historical   |1503    |3.8147             |270.3       |140            |3             |22            |
|Mystery      |2220    |3.8081             |396.6       |184            |4             |8             |
|Horror       |1257    |3.7835             |320.5       |167            |5             |15            |
|Psychological|2375    |3.7769             |267.6       |130            |6             |23            |
|Sports       |293     |3.7612             |235.1       |126    

## 7. Mencoba model terlatih

Inferensi tidak memakai Spark sama sekali — artefak dimuat langsung dari `Model/artifacts/`
agar responsnya milidetik.

In [10]:
from src.common.model_registry import artifacts_available, load_registry

sinopsis = (
    "Setelah dikhianati sekutunya dan tewas di puncak menara, seorang pendekar "
    "terbangun kembali sepuluh tahun lebih muda dan berniat menguasai teknik "
    "pedang terlarang sebelum musuh lamanya bangkit."
)
tags = "Time Travel, Weak to Strong, Cultivation, Male Protagonist, Revenge"

if not artifacts_available():
    print("Artefak belum ada. Latih dulu: python Model/train.py")
else:
    registry = load_registry()
    for name in registry.genre_model_names:
        info = registry.genre_model_info(name)
        hasil = registry.predict_genres(sinopsis, tags=tags, model=name, top_k=8)
        terpilih = hasil.loc[hasil["predicted"], "genre"].tolist()
        print(f"{info['label']:<22} macro-F1 {info['f1_macro']:.4f} -> {', '.join(terpilih)}")

2026-10-15 08:42:52 | INFO     | model_registry         | Registry dimuat: 39,660 novel, embedding (39660, 200), 35 genre, 4 model genre.


Ensemble SVM + Random Forest macro-F1 0.6121 -> Action, Martial Arts, Adventure, Fantasy, Xuanhuan, Xianxia, Shounen, Wuxia
Logistic Regression    macro-F1 0.6027 -> Action, Martial Arts, Adventure, Xuanhuan, Fantasy, Xianxia, Shounen, Wuxia
Linear SVM             macro-F1 0.6093 -> Action, Martial Arts, Adventure, Fantasy, Xuanhuan, Xianxia, Shounen
Random Forest          macro-F1 0.5522 -> Action, Adventure, Fantasy, Martial Arts, Xuanhuan, Xianxia


In [11]:
registry.recommend_by_text(tags, k=5)[
    ["title", "similarity", "primary_genre", "rating_score", "chapter_count"]
]

,title,similarity,primary_genre,rating_score,chapter_count
0,Mortal Immortal,0.6352,Fantasy,3.0,328.0
1,The Final Protectors,0.5584,Fantasy,3.9,89.0
2,American Justice,0.5523,Drama,2.0,358.0
3,I Became the Archmage That Transcendents Are O...,0.5299,Adventure,3.2,2.0
4,A NEET Otaku Can also be a Light Novel Writer!,0.5292,Comedy,2.6,NaN


## 8. Bukti Big Data: Variety, HDFS, MapReduce, Streaming

Bagian ini memeriksa LANGSUNG stack yang sedang hidup (WebHDFS, YARN REST API, keluaran
MapReduce di HDFS, checkpoint streaming) — prinsip yang sama dengan `bigdata_audit.py`:
bukti terukur, bukan klaim dari dokumentasi. Detail lengkap tiap konsep:
[`Doc/generated/bigdata_concept_audit.md`](../../Doc/generated/bigdata_concept_audit.md).

### 8.1 Variety — tiga berkas fisik per kelas format

In [12]:
import json as _json

manifest_path = settings.paths.project_root / "Dataset" / "variety" / "manifest.json"
manifest = _json.loads(manifest_path.read_text(encoding="utf-8"))
print(f"Sumber : {manifest['source']} ({manifest['source_rows']:,} baris)")
for f in manifest["files"]:
    rec = f.get("records")
    rec = f"{rec:,} baris" if rec else ""
    print(f"  {f['class']:<18} {f['format']:<6} {f['path']:<45} "
          f"{f['size_bytes'] / 1_048_576:8.2f} MB  {rec}")

Sumber : s3a://clean-data/novels_clean (39,668 baris)
  terstruktur        csv    structured/novel_metrics.csv                      4.06 MB  39,668 baris
  semi-terstruktur   jsonl  semistructured/novel_labels.jsonl                28.04 MB  39,668 baris
  tidak-terstruktur  txt    unstructured/synopsis/synopsis-0000.txt           7.06 MB  
  tidak-terstruktur  txt    unstructured/synopsis/synopsis-0001.txt           9.27 MB  
  tidak-terstruktur  txt    unstructured/synopsis/synopsis-0002.txt          10.61 MB  
  tidak-terstruktur  txt    unstructured/synopsis/synopsis-0003.txt          10.64 MB  
  referensi          txt    _ref/stopwords.txt                                0.00 MB  


### 8.2 HDFS — blok tersebar & berreplika di dua DataNode

Bukan sekadar 'file ada di HDFS' — ini menunjukkan blok fisiknya benar-benar tersebar
di `worker1`/`worker2` sesuai faktor replikasi.

In [13]:
import urllib.request


def webhdfs(path, op, **params):
    url = f"{settings.hdfs.webhdfs_url}/webhdfs/v1{path}?op={op}"
    for k, v in params.items():
        url += f"&{k}={v}"
    with urllib.request.urlopen(url, timeout=10) as resp:
        return _json.loads(resp.read())


listing = webhdfs("/novel/structured", "LISTSTATUS")
one_file = listing["FileStatuses"]["FileStatus"][0]
file_path = f"/novel/structured/{one_file['pathSuffix']}"
loc = webhdfs(file_path, "GET_BLOCK_LOCATIONS")
blocks = loc["LocatedBlocks"]["locatedBlocks"]
print(f"Berkas   : {file_path} ({one_file['length']:,} byte, replikasi {one_file['replication']})")
for b in blocks:
    hosts = [l.get("hostName") for l in b["locations"]]
    print(f"  Blok {b['block']['blockId']}: berreplika di {hosts}")

Berkas   : /novel/structured/novel_metrics.csv (4,300,991 byte, replikasi 2)
  Blok 1073741995: berreplika di ['worker1', 'worker2']
  Blok 1073741996: berreplika di ['worker1', 'worker2']


### 8.3 MapReduce — keluaran dibaca langsung dari HDFS, dicocokkan dengan Spark

Job `wordcount` (Hadoop Streaming, `src/mapreduce/wordcount_mapper.py`) menghitung token
sinopsis dari berkas TXT di HDFS. Di sini hasilnya dibaca ulang dan dicocokkan LANGSUNG
dengan hitungan Spark atas `novels_clean` yang sudah dimuat di atas — dua mesin, dua jalur
baca, satu jawaban.

In [14]:
mr_words = spark.read.text("hdfs://namenode:8020/novel/output/wordcount")
mr_count = mr_words.count()

from pyspark.sql import functions as F

spark_count = novels.select(F.explode("synopsis_tokens").alias("w")).select("w").distinct().count()

print(f"Kata unik hasil MapReduce (HDFS)     : {mr_count:,}")
print(f"Kata unik hasil Spark (novels_clean) : {spark_count:,}")
print("COCOK" if mr_count == spark_count else
      "TIDAK COCOK — jalankan scripts/mapreduce/run_jobs.sh atas data terbaru")

mr_words.select(F.split(F.col("value"), "\t").alias("kv")) \
    .select(F.col("kv")[0].alias("kata"), F.col("kv")[1].cast("int").alias("frekuensi")) \
    .orderBy(F.col("frekuensi").desc()) \
    .show(10, truncate=False)

Kata unik hasil MapReduce (HDFS)     : 62,416
Kata unik hasil Spark (novels_clean) : 62,412
TIDAK COCOK — jalankan scripts/mapreduce/run_jobs.sh atas data terbaru


+------+---------+
|kata  |frekuensi|
+------+---------+
|one   |25836    |
|world |25462    |
|life  |20946    |
|even  |14859    |
|like  |14535    |
|day   |13676    |
|time  |12154    |
|love  |11820    |
|family|11381    |
|years |11260    |
+------+---------+
only showing top 10 rows



### 8.4 YARN — NodeManager sehostname dengan DataNode (syarat data locality)

In [15]:
nodes_resp = urllib.request.urlopen("http://resourcemanager:8088/ws/v1/cluster/nodes", timeout=10)
nodes = _json.loads(nodes_resp.read())
for n in nodes["nodes"]["node"]:
    print(f"  {n['nodeHostName']:<10} state={n['state']:<10} containers={n['numContainers']}")

  worker2    state=RUNNING    containers=0
  worker1    state=RUNNING    containers=0


### 8.5 Streaming — checkpoint membuktikan ingest inkremental (exactly-once)

Kalau bagian ini bilang tabel belum ada, jalankan dulu demonya (lihat perintah di keluaran sel).

In [16]:
stream_out_uri = settings.minio.s3a_uri(settings.minio.bucket_curated, "streaming/novel_metrics_incremental")
try:
    stream_df = spark.read.parquet(stream_out_uri)
    n_stream = stream_df.count()
    ckpt_objs = list_objects(settings.minio.bucket_curated, "streaming/_checkpoint/offsets/")
    print(f"Baris di tabel streaming       : {n_stream:,}")
    print(f"Micro-batch tercatat (offset)  : {len(ckpt_objs)}")
except Exception:
    print("Tabel streaming belum ada — jalankan dulu:")
    print("  docker compose exec pipeline python scripts/streaming/simulate_landing.py")
    print("  docker compose exec pipeline python -m src.phases.streaming_ingest")

Baris di tabel streaming       : 1,000
Micro-batch tercatat (offset)  : 2


## 9. Menutup session

Jalankan sebelum menutup notebook agar slot executor di cluster dilepas —
kalau tidak, job Fase 1–4 berikutnya bisa antre menunggu sumber daya.

In [17]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
